# CZ Benchmarks Phase 2: Embedding Combination

This notebook implements Phase 2 of the CZ benchmarks evaluation: combining GenePT 1024d embeddings with foundation model embeddings (scGPT 512d and Transcriptformer 2048d).

Based on `specs/cz_benchmarks_evaluation_spec.md`

In [1]:
# Setup notebook environment
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

# Add parent directory to path
repo_dir = Path.cwd().parent
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print(f"Repository directory: {repo_dir}")
data_dir = repo_dir / "data"
print(f"Data directory: {data_dir}")

Repository directory: /Users/rj/personal/GenePT-tools
Data directory: /Users/rj/personal/GenePT-tools/data


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
TISSUE_TYPES = ['Blood', 'Bone_Marrow', 'Lung', 'Mammary', 'Thymus']
EMBEDDING_DIR = data_dir / 'cz_benchmark' / 'embeddings'
OUTPUT_DIR = EMBEDDING_DIR / 'combined'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Embedding directory: {EMBEDDING_DIR}")
print(f"Will save combined embeddings to: {OUTPUT_DIR}")

Embedding directory: /Users/rj/personal/GenePT-tools/data/cz_benchmark/embeddings
Will save combined embeddings to: /Users/rj/personal/GenePT-tools/data/cz_benchmark/embeddings/combined


## Embedding Combination Functions

In [3]:
def load_embeddings_for_tissue(tissue_type: str) -> dict:
    """
    Load all embedding types for a tissue and return aligned data.
    
    Returns dict with:
        - cell_ids: array of cell barcodes
        - genept: GenePT 1024d embeddings (n_cells, 1024)
        - scgpt: scGPT 512d embeddings (n_cells, 512)
        - transcriptformer: Transcriptformer 2048d embeddings (n_cells, 2048)
        - metadata: DataFrame with cell_type, donor_id, etc.
    """
    print(f"\n=== Loading embeddings for {tissue_type} ===")
    
    # 1. Load GenePT embeddings (has cell IDs in index)
    genept_path = EMBEDDING_DIR / 'genept_1024d' / f'genept_1024d_{tissue_type}_embeddings.parquet'
    genept_df = pd.read_parquet(genept_path)
    print(f"GenePT shape: {genept_df.shape}")
    
    # Extract cell IDs from index
    cell_ids = genept_df.index.values
    
    # Extract GenePT embeddings
    genept_cols = [col for col in genept_df.columns if col.startswith('genept_')]
    genept_embeddings = genept_df[genept_cols].values
    
    # Extract metadata
    metadata_cols = [col for col in genept_df.columns if not col.startswith('genept_')]
    metadata_df = genept_df[metadata_cols].copy()
    
    print(f"  - Cell IDs: {len(cell_ids)}")
    print(f"  - GenePT dimensions: {genept_embeddings.shape[1]}")
    
    # 2. Load scGPT embeddings (has cell_id column)
    scgpt_path = EMBEDDING_DIR / 'scgpt' / f'scgpt_{tissue_type}_embeddings.parquet'
    scgpt_df = pd.read_parquet(scgpt_path)
    print(f"scGPT shape: {scgpt_df.shape}")
    
    # Align scGPT by cell_id
    scgpt_df = scgpt_df.set_index('cell_id').loc[cell_ids].reset_index(drop=True)
    scgpt_cols = [col for col in scgpt_df.columns if col.startswith('embedding_')]
    scgpt_embeddings = scgpt_df[scgpt_cols].values
    print(f"  - scGPT dimensions: {scgpt_embeddings.shape[1]}")
    
    # 3. Load Transcriptformer embeddings (no cell_id, assume same order)
    tf_path = EMBEDDING_DIR / 'transcriptformer' / f'transcriptformer_tf_metazoa_{tissue_type}_embeddings.parquet'
    tf_df = pd.read_parquet(tf_path)
    print(f"Transcriptformer shape: {tf_df.shape}")
    
    # Extract Transcriptformer embeddings (assuming same order as scGPT/GenePT)
    tf_cols = [col for col in tf_df.columns if col.startswith('emb_')]
    transcriptformer_embeddings = tf_df[tf_cols].values
    print(f"  - Transcriptformer dimensions: {transcriptformer_embeddings.shape[1]}")
    
    # Verify all have same number of cells
    assert genept_embeddings.shape[0] == scgpt_embeddings.shape[0] == transcriptformer_embeddings.shape[0], \
        f"Cell count mismatch: GenePT={genept_embeddings.shape[0]}, scGPT={scgpt_embeddings.shape[0]}, TF={transcriptformer_embeddings.shape[0]}"
    
    return {
        'cell_ids': cell_ids,
        'genept': genept_embeddings,
        'scgpt': scgpt_embeddings,
        'transcriptformer': transcriptformer_embeddings,
        'metadata': metadata_df
    }

def create_combined_embeddings(embeddings_dict: dict, tissue_type: str) -> dict:
    """
    Create all embedding combinations.
    
    Returns dict with keys:
        - genept_1024d: GenePT alone (1024d)
        - genept_scgpt: GenePT + scGPT (1536d)
        - genept_transcriptformer: GenePT + Transcriptformer (3072d)
    """
    print(f"\n=== Creating embedding combinations for {tissue_type} ===")
    
    genept = embeddings_dict['genept']
    scgpt = embeddings_dict['scgpt']
    transcriptformer = embeddings_dict['transcriptformer']
    
    combinations = {
        'genept_1024d': genept,
        'genept_scgpt': np.hstack([genept, scgpt]),
        'genept_transcriptformer': np.hstack([genept, transcriptformer])
    }
    
    for name, emb in combinations.items():
        print(f"  - {name}: {emb.shape}")
    
    return combinations

def save_combined_embeddings(embeddings_dict: dict, combinations: dict, tissue_type: str):
    """
    Save combined embeddings with metadata to parquet files.
    """
    print(f"\n=== Saving combined embeddings for {tissue_type} ===")
    
    cell_ids = embeddings_dict['cell_ids']
    metadata = embeddings_dict['metadata']
    
    for name, emb_array in combinations.items():
        # Create DataFrame with embeddings
        n_dims = emb_array.shape[1]
        emb_df = pd.DataFrame(
            emb_array,
            index=cell_ids,
            columns=[f'{name}_{i}' for i in range(n_dims)]
        )
        
        # Add metadata
        for col in metadata.columns:
            emb_df[col] = metadata[col].values
        
        # Save
        output_path = OUTPUT_DIR / f'{name}_{tissue_type}_embeddings.parquet'
        emb_df.to_parquet(output_path)
        print(f"  - Saved {name}: {output_path.name} ({emb_df.shape})")
    
    return True

## Process All Tissues

In [4]:
# Process each tissue
results_summary = []

for tissue_type in tqdm(TISSUE_TYPES, desc="Processing tissues"):
    try:
        # Load all embeddings
        embeddings_dict = load_embeddings_for_tissue(tissue_type)
        
        # Create combinations
        combinations = create_combined_embeddings(embeddings_dict, tissue_type)
        
        # Save combined embeddings
        save_combined_embeddings(embeddings_dict, combinations, tissue_type)
        
        # Track results
        results_summary.append({
            'tissue': tissue_type,
            'n_cells': len(embeddings_dict['cell_ids']),
            'genept_1024d_dims': combinations['genept_1024d'].shape[1],
            'genept_scgpt_dims': combinations['genept_scgpt'].shape[1],
            'genept_transcriptformer_dims': combinations['genept_transcriptformer'].shape[1],
            'status': 'success'
        })
        
    except Exception as e:
        print(f"ERROR processing {tissue_type}: {str(e)}")
        import traceback
        traceback.print_exc()
        
        results_summary.append({
            'tissue': tissue_type,
            'status': 'failed',
            'error': str(e)
        })
        continue

print(f"\n=== Processing Complete ===")
print(f"Successfully processed {sum(1 for r in results_summary if r['status'] == 'success')} out of {len(TISSUE_TYPES)} tissues")

Processing tissues:   0%|          | 0/5 [00:00<?, ?it/s]


=== Loading embeddings for Blood ===
GenePT shape: (17802, 1027)
  - Cell IDs: 17802
  - GenePT dimensions: 1024
scGPT shape: (17802, 516)
  - scGPT dimensions: 512
Transcriptformer shape: (17802, 2049)
  - Transcriptformer dimensions: 2048

=== Creating embedding combinations for Blood ===
  - genept_1024d: (17802, 1024)
  - genept_scgpt: (17802, 1536)
  - genept_transcriptformer: (17802, 3072)

=== Saving combined embeddings for Blood ===
  - Saved genept_1024d: genept_1024d_Blood_embeddings.parquet ((17802, 1027))
  - Saved genept_scgpt: genept_scgpt_Blood_embeddings.parquet ((17802, 1539))
  - Saved genept_transcriptformer: genept_transcriptformer_Blood_embeddings.parquet ((17802, 3075))

=== Loading embeddings for Bone_Marrow ===
GenePT shape: (8045, 1027)
  - Cell IDs: 8045
  - GenePT dimensions: 1024
scGPT shape: (8045, 516)
  - scGPT dimensions: 512
Transcriptformer shape: (8045, 2049)
  - Transcriptformer dimensions: 2048

=== Creating embedding combinations for Bone_Marrow =

## Summary and Validation

In [5]:
# Create summary table
summary_df = pd.DataFrame(results_summary)
print("\n=== Embedding Combination Summary ===")
print(summary_df.to_string(index=False))

# List all generated files
print(f"\n=== Generated Files ===")
print(f"Output directory: {OUTPUT_DIR}")
all_files = list(OUTPUT_DIR.glob('*.parquet'))
print(f"Total files: {len(all_files)}")

for file_path in sorted(all_files):
    size_mb = file_path.stat().st_size / 1024 / 1024
    print(f"  - {file_path.name} ({size_mb:.1f} MB)")

# Validate one example
if len(all_files) > 0:
    print(f"\n=== Validation: Loading sample file ===")
    sample_file = all_files[0]
    sample_df = pd.read_parquet(sample_file)
    print(f"File: {sample_file.name}")
    print(f"Shape: {sample_df.shape}")
    print(f"Columns: {list(sample_df.columns)[:10]} ... {list(sample_df.columns)[-3:]}")
    
    # Check for NaN/Inf
    emb_cols = [col for col in sample_df.columns if any(x in col for x in ['genept_', 'scgpt', 'transcriptformer'])]
    if sample_df[emb_cols].isna().any().any():
        print("WARNING: NaN values found in embeddings!")
    elif np.isinf(sample_df[emb_cols].values).any():
        print("WARNING: Inf values found in embeddings!")
    else:
        print("✓ No NaN/Inf values detected")

print("\nPhase 2 Complete: Embedding combinations created for all tissues!")


=== Embedding Combination Summary ===
     tissue  n_cells  genept_1024d_dims  genept_scgpt_dims  genept_transcriptformer_dims  status
      Blood    17802               1024               1536                          3072 success
Bone_Marrow     8045               1024               1536                          3072 success
       Lung    11716               1024               1536                          3072 success
    Mammary    18539               1024               1536                          3072 success
     Thymus     9933               1024               1536                          3072 success

=== Generated Files ===
Output directory: /Users/rj/personal/GenePT-tools/data/cz_benchmark/embeddings/combined
Total files: 15
  - genept_1024d_Blood_embeddings.parquet (172.6 MB)
  - genept_1024d_Bone_Marrow_embeddings.parquet (76.4 MB)
  - genept_1024d_Lung_embeddings.parquet (112.4 MB)
  - genept_1024d_Mammary_embeddings.parquet (179.7 MB)
  - genept_1024d_Thymus_embeddin